In [0]:
# Bronze Layer - Raw data ingestion
# Reads files from Data Lake and saves them as Delta tables without transformation

In [0]:
# Configuration
storage_account_name = dbutils.secrets.get(scope="etl_sales_scope", key="STORAGE_ACCOUNT_NAME")
storage_account_key = dbutils.secrets.get(scope="etl_sales_scope", key="STORAGE_ACCOUNT_KEY")
container_name = dbutils.secrets.get(scope="etl_sales_scope", key="AZURE_CONTAINER_NAME")
new_directory = dbutils.secrets.get(scope="etl_sales_scope", key="NEW_DATA_DIRECTORY")
processed_directory = dbutils.secrets.get(scope="etl_sales_scope", key="PROCESSED_DATA_DESTINATION")
error_directory = dbutils.secrets.get(scope="etl_sales_scope", key="ERROR_DATA_DESTINATION")

# Configure access to Data Lake
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key,
)

print("Configuration loaded successfully")

In [0]:
# List new files to process
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"
new_path = f"{base_path}/{new_directory}"
processed_path = f"{base_path}/{processed_directory}"
error_path = f"{base_path}/{error_directory}"

new_files = dbutils.fs.ls(new_path)
new_files = [f for f in new_files if not f.name.endswith('/')]

print(f"Found {len(new_files)} new files to process")
for f in new_files:
    print(f"📄 {f.name}")

In [0]:
# Read and save to Bronze Delta table

from pyspark.sql.functions import current_timestamp, input_file_name

if not new_files:
    print("No new files to process")
    dbutils.notebook.exit("No new files")

for file in new_files:
    try:
        print(f"Processing: {file.name}")
        
        # Read CSV
        df_raw = spark.read.csv(
            file.path,
            header=True,
            inferSchema=True
        )
        
        # Add audit columns
        df_raw = df_raw \
            .withColumn("_ingested_at", current_timestamp()) \
            .withColumn("_source_file", input_file_name())
        
        print(f"Shape: {df_raw.count()} rows x {len(df_raw.columns)} columns")
        
        # Save as Delta table in Bronze layer
        df_raw.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .save(f"{base_path}/bronze/sales")
        
        print(f"{file.name} saved to Bronze layer")
        
        # Move file to processed
        dbutils.fs.mv(
            file.path,
            f"{processed_path}/{file.name}"
        )
        print(f"{file.name} moved to processed/")

    except Exception as e:
        print(f"Error processing {file.name}: {e}")
        
        # Move file to error
        dbutils.fs.mv(
            file.path,
            f"{error_path}/{file.name}"
        )
        print(f"{file.name} moved to error/")
        raise

In [0]:
# Verify Bronze table

df_bronze = spark.read.format("delta").load(f"{base_path}/bronze/sales")
print(f"Bronze table total rows: {df_bronze.count()}")
display(df_bronze)